In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
print("TensorFlow version:", tf.__version__)
# Load the TensorBoard notebook extension
%load_ext tensorboard
import datetime
import shutil

TensorFlow version: 2.18.0


In [2]:
%reload_ext tensorboard
from tensorflow.keras import Model
from pathlib import Path
import pandas as pd

In [10]:
BATCH_SIZE = 64   # 64
BUFFER_SIZE = 512   # 512
LEARNING_RATE = 0.001 # 0.001  
EPOCHS = 4000 # 2000
# best results 
# - Epoch 2000, Loss: 0.4995439648628235, Accuracy: 0.5163321495056152, Test Loss: 0.43891793489456177, Test MAE: 0.47739431262016296
# - Epoch 4000, Loss: 0.45906156301498413, Accuracy: 0.49423515796661377, Test Loss: 0.42648589611053467, Test MAE: 0.47011545300483704


In [4]:
input_dir = Path('./data/prepared')
logs_path = Path('./data/logs')
if logs_path.exists():
  shutil.rmtree(logs_path) # удаляем, если существует /logs
logs_path.mkdir(parents=True)

X_train_name = input_dir / 'X_train.csv'
y_train_name = input_dir / 'y_train.csv'
X_test_name = input_dir / 'X_test.csv'
y_test_name = input_dir / 'y_test.csv'

X_train = pd.read_csv(X_train_name)
y_train = pd.read_csv(y_train_name)
X_test = pd.read_csv(X_test_name)
y_test = pd.read_csv(y_test_name)

X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()

# print(X_train.isnull().sum())
# print(y_train.isnull().sum())
# print(X_train.dtypes)  # Типы данных в X_train
# print(y_train.dtypes)  # Типы данных в y_train

# Проблема в one_hot_encoding - исправим обратно так
one_hot_columns = ['type_Movie', 'type_Music', 'type_ONA', 'type_OVA', 'type_Special', 'type_TV']
X_train['type_encoded'] = X_train[one_hot_columns].idxmax(axis=1).str.split('_').str[1]
X_test['type_encoded'] = X_test[one_hot_columns].idxmax(axis=1).str.split('_').str[1]
label_mapping = {label: idx for idx, label in enumerate(sorted(X_train['type_encoded'].unique()))}
X_train['type_encoded'] = X_train['type_encoded'].map(label_mapping)
X_test['type_encoded'] = X_test['type_encoded'].map(label_mapping)

X_train = X_train.drop(columns=one_hot_columns)
X_test = X_test.drop(columns=one_hot_columns)
y_train = pd.read_csv(y_train_name)
y_test = pd.read_csv(y_test_name)

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

In [5]:
@tf.keras.utils.register_keras_serializable() #  Декоратор позволяет сериализовать и десериализовать модель для сохранения и загрузки.
class SomeModel(Model):
    def __init__(self, neurons_cnt=64, **kwargs):
        super(SomeModel, self).__init__(**kwargs)
        self.neurons_cnt = neurons_cnt  # Сохраняем значение параметра для конфигурации
        self.d_in = Dense(27, activation='relu')
        self.d1 = Dense(neurons_cnt, activation='relu')
        self.d2 = Dense(neurons_cnt, activation='relu')
        self.d3 = Dense(neurons_cnt, activation='relu')
        self.d_out = Dense(1)

    def call(self, x):
        x = self.d_in(x)
        x = self.d1(x)
        x = self.d2(x)
        x = self.d3(x)
        return self.d_out(x)
         
    def build(self, input_shape): # надо явно определить для построения
        super(SomeModel, self).build(input_shape)
        
    def get_config(self): 
        # Возвращаем параметры модели, включая кастомные
        config = super(SomeModel, self).get_config()
        config.update({
            "neurons_cnt": self.neurons_cnt  # Добавляем кастомный параметр в конфигурацию
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Создаём экземпляр класса из конфигурации
        return cls(**config)

In [6]:
# Create an instance of the model
model = SomeModel(neurons_cnt=32) # 32
model.build(input_shape=(None, 31))  # 27

In [7]:
loss_object = tf.keras.losses.MeanSquaredError() # что? 
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.MeanAbsoluteError(name='train_mae')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.MeanAbsoluteError(name='test_mae')

In [8]:
@tf.function
def train_step(input_vector, labels):
  with tf.GradientTape() as tape:
    # training=True is only needed if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    predictions = model(input_vector, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)
  train_accuracy(labels, predictions)

@tf.function
def test_step(input_vector, labels):
  # training=False is only needed if there are layers with different
  # behavior during training versus inference (e.g. Dropout).
  predictions = model(input_vector, training=False)
  t_loss = loss_object(labels, predictions)

  test_loss(t_loss)
  test_accuracy(labels, predictions)

In [9]:
from tensorflow import keras
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = logs_path / 'gradient_tape' / current_time / 'train'
train_log_dir.mkdir(exist_ok=True, parents=True)
test_log_dir = logs_path / 'gradient_tape' / current_time / 'test'
test_log_dir.mkdir(exist_ok=True, parents=True)
train_summary_writer = tf.summary.create_file_writer(str(train_log_dir))
test_summary_writer = tf.summary.create_file_writer(str(test_log_dir))

logdir=logs_path / "fit" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
logdir.mkdir(exist_ok=True, parents=True)
fit_summary_writer = tf.summary.create_file_writer(str(logdir))

tf.summary.trace_on(graph=True, profiler=True, profiler_outdir=str(logdir))

for epoch in range(EPOCHS):
  # Reset the metrics at the start of the next epoch
  for (x_train, y_train) in train_ds:

    with fit_summary_writer.as_default():
      train_step(x_train, y_train)


  with train_summary_writer.as_default():
    tf.summary.scalar('loss', train_loss.result(), step=epoch)
    tf.summary.scalar('accuracy', train_accuracy.result(), step=epoch)

  for (x_test, y_test) in test_ds:
    test_step(x_test, y_test)

  with test_summary_writer.as_default():
    tf.summary.scalar('loss', test_loss.result(), step=epoch)
    tf.summary.scalar('mae', test_accuracy.result(), step=epoch)

  template = 'Epoch {}, Loss: {}, Accuracy: {}, Test Loss: {}, Test MAE: {}'
  print (template.format(epoch+1,
                         train_loss.result(),
                         train_accuracy.result(),
                         test_loss.result(),
                         test_accuracy.result()))

  # Reset metrics every epoch
  train_loss.reset_state()
  test_loss.reset_state()
  train_accuracy.reset_state()
  test_accuracy.reset_state()

with fit_summary_writer.as_default():
  tf.summary.trace_export(
      name="my_func_trace",
      step=0,
      profiler_outdir=str(logdir)
  )

Epoch 1, Loss: 122150.6640625, Accuracy: 76.31613159179688, Test Loss: 23.742576599121094, Test MAE: 4.651759147644043
Epoch 2, Loss: 22.399152755737305, Accuracy: 4.304502964019775, Test Loss: 22.066022872924805, Test MAE: 4.126452445983887
Epoch 3, Loss: 20.9846248626709, Accuracy: 4.156251907348633, Test Loss: 20.337862014770508, Test MAE: 4.156198024749756
Epoch 4, Loss: 19.59874725341797, Accuracy: 4.0161027908325195, Test Loss: 18.718833923339844, Test MAE: 4.023251533508301
Epoch 5, Loss: 17.686809539794922, Accuracy: 3.8135809898376465, Test Loss: 16.85208511352539, Test MAE: 3.67222261428833
Epoch 6, Loss: 15.857671737670898, Accuracy: 3.6022536754608154, Test Loss: 14.98953914642334, Test MAE: 3.479458808898926
Epoch 7, Loss: 14.14761734008789, Accuracy: 3.4000673294067383, Test Loss: 13.154389381408691, Test MAE: 3.2955665588378906
Epoch 8, Loss: 12.369699478149414, Accuracy: 3.1692001819610596, Test Loss: 11.427946090698242, Test MAE: 3.1066253185272217
Epoch 9, Loss: 10.54